In [11]:
!pip install onnxruntime tokenizers numpy tqdm minsearch gitsource huggingface-hub jupyter


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: C:\Users\borge\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


# Q1

In [12]:
import numpy as np
from tqdm.auto import tqdm
from llmzoomcamponnx.embeder import Embedder

model = Embedder()

from ingest import load_faq_data

documents = load_faq_data()
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode_batch(batch)
    vectors.extend(batch_vectors)

X = np.array(vectors)

query = "How does approximate nearest neighbor search work?"
v_query = model.encode(query) 

scores = X.dot(v_query)

best_idx = np.argmax(scores)
print("Q1 Answer:", v_query[0])

  0%|          | 0/27 [00:00<?, ?it/s]

Q1 Answer: -0.02058203437252893


# Q2

In [ ]:
from gitsource import GithubRepositoryDataReader
import numpy as np

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

target_path = "02-vector-search/lessons/07-sqlitesearch-vector.md"
target_doc = None

for doc in documents:
    if doc.get("filename") == target_path:
        target_doc = doc
        break

if target_doc is None:
    raise ValueError(f"Could not find {target_path} in the downloaded files!")

doc_text = target_doc["content"]

v_doc = model.encode(doc_text)

similarity = v_doc.dot(v_query)

print(f"Cosine Similarity: {similarity:.2f}")

Cosine Similarity: 0.36


# Q3

In [ ]:
from gitsource import chunk_documents
import numpy as np

chunks = chunk_documents(documents, size=2000, step=1000)

chunk_texts = [chunk["content"] for chunk in chunks]

chunk_vectors = model.encode_batch(chunk_texts)

X = np.array(chunk_vectors)

scores = X.dot(v_query)

highest_score_idx = np.argmax(scores)

winning_chunk = chunks[highest_score_idx]
print(f"Highest Score: {scores[highest_score_idx]:.4f}")
print(f"Filename: {winning_chunk['filename']}")

Highest Score: 0.6489
Filename: 02-vector-search/lessons/07-sqlitesearch-vector.md


# Q4

In [ ]:
from minsearch import VectorSearch

vector_index = VectorSearch(keyword_fields=["filename"])

vector_index.fit(X, chunks)

query_q4 = "What metric do we use to evaluate a search engine?"
v_query_q4 = model.encode(query_q4)

results = vector_index.search(v_query_q4, num_results=5)

print(f"Top result filename: {results[0]['filename']}")

Top result filename: 04-evaluation/lessons/05-search-metrics.md


# Q5


In [21]:
from minsearch import Index

text_index = Index(text_fields=["content"], keyword_fields=["filename"])
text_index.fit(chunks)

query_q5 = "How do I store vectors in PostgreSQL?"
v_query_q5 = model.encode(query_q5)

vector_results = vector_index.search(v_query_q5, num_results=5)
vector_filenames = set(res["filename"] for res in vector_results)

text_results = text_index.search(query_q5, num_results=5)
text_filenames = set(res["filename"] for res in text_results)

exclusive_vector_files = vector_filenames - text_filenames

print(list(exclusive_vector_files)[0])

02-vector-search/lessons/08-pgvector.md


# Q6

In [ ]:
query_q6 = "How do I give the model access to tools?"
v_query_q6 = model.encode(query_q6)

vector_results_q6 = vector_index.search(v_query_q6, num_results=5)
text_results_q6 = text_index.search(query_q6, num_results=5)

def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

# 4. Fuse the results and print the first ranked file
fused_results = rrf([vector_results_q6, text_results_q6])
print(fused_results[0]["filename"])

01-agentic-rag/lessons/13-function-calling.md
